In [1]:
REPO_URL  = "https://github.com/nardouhn/ai-translator-system.git"
!git clone {REPO_URL}

Cloning into 'ai-translator-system'...
remote: Enumerating objects: 11423, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 11423 (delta 0), reused 5 (delta 0), pack-reused 11411 (from 1)
Receiving objects: 100% (11423/11423), 231.42 MiB | 24.77 MiB/s, done.
Resolving deltas: 100% (3698/3698), done.


In [2]:
!pip install pyngrok flask-ngrok transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 25.4 MB/s eta 0:00:0000:0100:01


In [3]:
%cd /kaggle/working/ai-translator-system


/kaggle/working/ai-translator-system


In [4]:
!git fetch
!git checkout feature/rag-module

Branch 'feature/rag-module' set up to track remote branch 'feature/rag-module' from 'origin'.
Switched to a new branch 'feature/rag-module'


In [5]:
!pip install -q -r requirements.txt pyngrok flask accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 753.8 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 45.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 408.9/408.9 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 57.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.5 MB/s eta 0:00:00
ERROR: pip's depende

In [6]:
# translator_engine.cache.clear()

In [7]:
# =========================================================
# 1. INSTALL & SETUP MÔI TRƯỜNG (Chạy trong Kaggle/Colab)
# =========================================================
# !pip install -q pyngrok flask transformers accelerate bitsandbytes sentencepiece

import os
import sys
import torch
import threading
import time
import hashlib
import re
from collections import OrderedDict
from flask import Flask, request, jsonify
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from pyngrok import ngrok

# Thiết lập đường dẫn để import prompt_config từ repo đã clone
REPO_PATH = "/kaggle/working/ai-translator-system"
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

# Import module từ repo của bạn
try:
    from prompt_config import get_context_prompt, format_qwen_prompt
except ImportError:
    print("❌ Không tìm thấy prompt_config. Hãy đảm bảo đã clone repo và checkout đúng branch.")

# =========================================================
# 2. CONFIGURATION
# =========================================================
NGROK_AUTH_TOKEN = "3Cex6EqkiCGznGVeaoTgBrgdC2F_5ypVQwykwACcLCRUmgJPK"
MODEL_ID = "ltyen05/qwen-domain-translator"
ALLOWED_DOMAINS = ["general", "it", "finance", "medical"]
MAX_CACHE_SIZE = 1000  # Giới hạn 1000 câu dịch gần nhất để tránh tràn RAM
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================================================
# 3. LOAD MODEL VỚI TỐI ƯU HÓA 4-BIT (CỰC NHANH TRÊN GPU)
# =========================================================
print(f"🚀 Đang tải model {MODEL_ID} trên {DEVICE}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

def format_qwen_prompt(user_input, context, domain):
    domain_instructions = {
        "it": "Sử dụng thuật ngữ chuyên ngành CNTT.",
        "finance": "Sử dụng thuật ngữ chuyên ngành Tài chính",
        "medical": "Sử dụng thuật ngữ Y tế .",
        "general": "Dịch theo văn phong phổ thông, dễ hiểu."
    }
    instruction = domain_instructions.get(domain, domain_instructions["general"])
    
    return f"""<|im_start|>system
Bạn là chuyên gia dịch thuật {domain.upper()}. 
Yêu cầu: {instruction}
Ngữ cảnh: {context}
Chỉ trả về bản dịch.<|im_end|>
<|im_start|>user
Dịch sang tiếng Việt phong cách {domain}: "{user_input}"<|im_end|>
<|im_start|>assistant
"""


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.eval()
print("✅ Model đã sẵn sàng!")

# =========================================================
# 4. HỆ THỐNG CACHE & XỬ LÝ VĂN BẢN
# =========================================================
class TranslationSystem:
    def __init__(self, max_cache=MAX_CACHE_SIZE):
        self.cache = OrderedDict()
        self.max_cache = max_cache

    def get_hash(self, text, domain):
        return hashlib.md5(f"{domain}:{text}".encode('utf-8')).hexdigest()

    def split_text(self, text, max_chars=800):
        """Chia văn bản dài thành các đoạn nhỏ dựa trên dấu câu."""
        sentences = re.split(r'(?<=[.!?])\s+', text)
        chunks = []
        current = ""
        for s in sentences:
            if len(current) + len(s) < max_chars:
                current += s + " "
            else:
                chunks.append(current.strip())
                current = s + " "
        if current:
            chunks.append(current.strip())
        return chunks

    def translate(self, text, domain):
        chunks = self.split_text(text)
        results = []
        any_new_inference = False

        for chunk in chunks:
            key = self.get_hash(chunk, domain)
            
            if key in self.cache:
                # Move to end (LRU policy)
                self.cache.move_to_end(key)
                results.append(self.cache[key])
            else:
                # Inference thực tế trên GPU
                translated_chunk = self._inference(chunk, domain)
                self.cache[key] = translated_chunk
                results.append(translated_chunk)
                any_new_inference = True
                
                # Duy trì kích thước cache
                if len(self.cache) > self.max_cache:
                    self.cache.popitem(last=False)

        return " ".join(results), not any_new_inference

    def _inference(self, chunk, domain):
        # Lấy context từ RAG module trong repo của bạn
        rag_context = get_context_prompt(user_input=chunk, domain=domain)
        
        # Tạo prompt chuẩn cho Qwen
        full_prompt = format_qwen_prompt(
            user_input=chunk,
            context=rag_context,
            domain=domain
        )

        inputs = tokenizer(full_prompt, return_tensors="pt").to(DEVICE)

        with torch.inference_mode(): # Tối ưu tốc độ vượt trội so với no_grad()
            outputs = model.generate(
                **inputs,
                max_new_tokens=1024,
                do_sample=True,
                temperature=0.1,    # Thấp để dịch chính xác thuật ngữ
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id
            )

        generated_ids = outputs[0][inputs.input_ids.shape[-1]:]
        return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

translator_engine = TranslationSystem()

# =========================================================
# 5. FLASK API & NGROK
# =========================================================
app = Flask(__name__)

@app.route("/translate", methods=["POST"])
def api_translate():
    data = request.get_json()
    text = data.get("text", "").strip()
    domain = data.get("domain", "general")

    if not text:
        return jsonify({"error": "Vui lòng nhập văn bản"}), 400
    if domain not in ALLOWED_DOMAINS:
        return jsonify({"error": "Domain không hợp lệ"}), 400

    start_time = time.time()
    try:
        output, is_cached = translator_engine.translate(text, domain)
        elapsed = time.time() - start_time
        
        return jsonify({
            "output": output,
            "domain": domain,
            "cached": is_cached,
            "time_seconds": round(elapsed, 3),
            "device": DEVICE
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route("/ui")
def ui():
    return """
    <html>
    <head>
        <title>AI Translator Pro</title>
        <style>
            body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; max-width: 800px; margin: 40px auto; line-height: 1.6; background: #f0f2f5; }
            .container { background: white; padding: 30px; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.1); }
            textarea { width: 100%; border: 1px solid #ddd; border-radius: 8px; padding: 12px; font-size: 16px; resize: vertical; }
            select, button { padding: 10px 20px; font-size: 16px; border-radius: 6px; border: 1px solid #ddd; cursor: pointer; }
            button { background: #007bff; color: white; border: none; transition: 0.3s; }
            button:hover { background: #0056b3; }
            #result { background: #f8f9fa; border-left: 5px solid #007bff; padding: 15px; margin-top: 20px; white-space: pre-wrap; min-height: 50px; }
            .meta { font-size: 12px; color: #666; margin-top: 10px; }
        </style>
    </head>
    <body>
        <div class="container">
            <h2>🌍 AI Domain Translator <small>(Optimized)</small></h2>
            <textarea id="text" rows="8" placeholder="Nhập văn bản tiếng Anh cần dịch..."></textarea>
            <div style="margin-top: 15px; display: flex; gap: 10px; align-items: center;">
                <select id="domain">
                    <option value="general">General</option>
                    <option value="it">Information Technology</option>
                    <option value="finance">Finance</option>
                    <option value="medical">Medical</option>
                </select>
                <button onclick="send()">Dịch Ngay</button>
            </div>
            <div id="result">Kết quả sẽ hiển thị ở đây...</div>
            <div id="meta" class="meta"></div>
        </div>
        <script>
            async function send() {
                const btn = document.querySelector("button");
                const resDiv = document.getElementById("result");
                const metaDiv = document.getElementById("meta");
                
                const text = document.getElementById("text").value;
                const domain = document.getElementById("domain").value;
                
                if(!text) return alert("Nhập chữ đã bạn ơi!");
                
                btn.disabled = true;
                resDiv.innerText = "⚡ Đang xử lý trên GPU (RAG + Inference)...";
                
                try {
                    const res = await fetch("/translate", {
                        method: "POST",
                        headers: {"Content-Type": "application/json"},
                        body: JSON.stringify({text, domain})
                    });
                    const data = await res.json();
                    resDiv.innerText = data.output || data.error;
                    metaDiv.innerText = `Thời gian: ${data.time_seconds}s | Cache: ${data.cached} | Device: ${data.device}`;
                } catch(e) {
                    resDiv.innerText = "Lỗi kết nối: " + e;
                } finally {
                    btn.disabled = false;
                }
            }
        </script>
    </body>
    </html>
    """

# =========================================================
# 6. RUN
# =========================================================
def start_ngrok():
    try:
        ngrok.kill()
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
        public_url = ngrok.connect(5000)
        print(f"\n🚀 TRUY CẬP UI TẠI: {public_url.public_url}/ui")
    except Exception as e:
        print(f"Lỗi Ngrok: {e}")

if __name__ == "__main__":
    start_ngrok()
    app.run(host="0.0.0.0", port=5000, use_reloader=False)

Loading Reranker model (ms-marco-MiniLM-L-6-v2) for multi-stage RAG...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

✅ Model đã sẵn sàng!
                                                                                                    
🚀 TRUY CẬP UI TẠI: https://liability-uncharted-identity.ngrok-free.dev/ui
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.19.2.2:5000
Press CTRL+C to quit
127.0.0.1 - - [05/May/2026 16:36:31] "GET /ui HTTP/1.1" 200 -
127.0.0.1 - - [05/May/2026 16:36:39] "POST /translate HTTP/1.1" 200 -
127.0.0.1 - - [05/May/2026 16:37:09] "POST /translate HTTP/1.1" 200 -
127.0.0.1 - - [05/May/2026 16:37:19] "POST /translate HTTP/1.1" 200 -
127.0.0.1 - - [05/May/2026 16:37:29] "POST /translate HTTP/1.1" 200 -
127.0.0.1 - - [05/May/2026 16:42:45] "POST /translate HTTP/1.1" 200 -
127.0.0.1 - - [05/May/2026 16:55:11] "POST /translate HTTP/1.1" 200 -
127.0.0.1 - - [05/May/2026 17:01:36] "GET /ui HTTP/1.1" 200 -
127.0.0.1 - - [05/May/2026 17:07:13] "POST /translate HTTP/1.1" 200 -
